# LangGraph — Продвинутые агенты

## Что такое LangGraph?

**LangGraph** — это библиотека для создания сложных агентов с явными графами состояний.

### Проблема с простыми агентами:

```python
# Простой агент (из предыдущего урока)
while not done:
    response = llm.invoke(messages)
    if response.tool_calls:
        result = execute_tool()
        messages.append(result)
    else:
        done = True
```

**Проблемы:**
- ❌ Нет явной логики переходов
- ❌ Сложно добавить ветвления
- ❌ Нет циклов с условиями
- ❌ Трудно отлаживать

### Решение — LangGraph:

```python
graph = StateGraph()
graph.add_node("think", think_node)
graph.add_node("action", action_node)
graph.add_edge("think", "action")
graph.add_conditional_edges(
    "action",
    should_continue,  # Функция решает: продолжить или завершить
    {
        "continue": "think",
        "end": END
    }
)
```

**Преимущества:**
- ✅ Явная структура workflow
- ✅ Легко добавлять ветвления и циклы
- ✅ Визуализация графа
- ✅ Проще отлаживать

### Java аналогия:

```java
// LangGraph — это как State Machine (конечный автомат)

// Простой агент — это while loop
while (!done) {
    result = process();
    if (shouldContinue(result)) {
        continue;
    } else {
        done = true;
    }
}

// LangGraph — это State Pattern с переходами
class StateMachine {
    Map<String, State> states;
    Map<String, Transition> transitions;
    
    void execute() {
        State current = states.get("start");
        while (current != null) {
            Result result = current.execute();
            Transition next = transitions.get(current.decide(result));
            current = next.getNextState();
        }
    }
}

// Похоже на:
// - Spring State Machine
// - Activiti/Camunda (BPMN workflow)
// - AWS Step Functions
```

## Основные концепции

### 1. State (Состояние)

**State** — данные которые передаются между узлами.

```python
from typing import TypedDict

class State(TypedDict):
    messages: list  # История диалога
    counter: int    # Счётчик итераций
    result: str     # Финальный результат
```

**Java аналогия:**
```java
// State — это как Context в паттерне State
class WorkflowContext {
    List<Message> messages;
    int counter;
    String result;
}
```

### 2. Node (Узел)

**Node** — функция которая обрабатывает состояние.

```python
def my_node(state: State) -> State:
    # Обработка состояния
    state["counter"] += 1
    return state
```

**Java аналогия:**
```java
// Node — это как обработчик состояния
interface StateHandler {
    WorkflowContext handle(WorkflowContext context);
}

class ThinkNode implements StateHandler {
    public WorkflowContext handle(WorkflowContext ctx) {
        ctx.counter++;
        return ctx;
    }
}
```

### 3. Edge (Ребро)

**Edge** — переход от одного узла к другому.

```python
# Обычное ребро (unconditional)
graph.add_edge("node_a", "node_b")  # Всегда переходит от A к B

# Условное ребро (conditional)
graph.add_conditional_edges(
    "node_a",
    decide_function,  # Функция решает куда перейти
    {
        "path1": "node_b",
        "path2": "node_c"
    }
)
```

**Java аналогия:**
```java
// Edge — это как переход в State Machine
class Transition {
    State from;
    State to;
    Predicate<Context> condition;  // Условие перехода
}
```

### 4. Граф переходов

```
START
  ↓
[Think]  ← Узел: LLM думает
  ↓
[Action] ← Узел: Выполняет инструмент
  ↓
Условие: Продолжить?
  ├─ Да → возврат к [Think]
  └─ Нет → END
```

## Установка

In [1]:
# Установим LangGraph
!pip install -q langgraph langchain langchain-openai langchain-core python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Загружаем API ключ
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

print(f"API Key загружен: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key загружен: sk-***...


## Пример 1: Простой граф с циклом

Создадим агента который считает до 5:

```
START → [Increment] → Условие: < 5?
             ↑            |
             └────Да──────┘
                  |
                 Нет
                  ↓
                 END
```

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# 1. Определяем состояние
class CounterState(TypedDict):
    counter: int
    max_count: int

# 2. Узел: Увеличивает счётчик
def increment_node(state: CounterState) -> CounterState:
    print(f"Счётчик: {state['counter']}")
    state["counter"] += 1
    return state

# 3. Функция решения: Продолжить или завершить?
def should_continue(state: CounterState) -> str:
    if state["counter"] < state["max_count"]:
        return "continue"  # Продолжаем
    else:
        return "end"  # Завершаем

# 4. Создаём граф
workflow = StateGraph(CounterState)

# Добавляем узел
workflow.add_node("increment", increment_node)

# Устанавливаем точку входа
workflow.set_entry_point("increment")

# Добавляем условное ребро
workflow.add_conditional_edges(
    "increment",
    should_continue,
    {
        "continue": "increment",  # Возвращаемся к increment
        "end": END  # Завершаем
    }
)

# Компилируем граф
app = workflow.compile()

print("Граф создан!")

Граф создан!


In [4]:
# Запускаем граф
initial_state = {
    "counter": 0,
    "max_count": 5
}

result = app.invoke(initial_state)

print("\n" + "="*50)
print(f"Финальное состояние: {result}")

Счётчик: 0
Счётчик: 1
Счётчик: 2
Счётчик: 3
Счётчик: 4

Финальное состояние: {'counter': 5, 'max_count': 5}


### Что произошло?

1. Граф начал с `counter=0`
2. Узел `increment` увеличил счётчик: `0 → 1`
3. `should_continue` проверил: `1 < 5` → "continue"
4. Вернулись к `increment`: `1 → 2`
5. Цикл повторялся пока `counter < 5`
6. Когда `counter = 5` → "end" → граф завершился

### Java аналогия:

```java
// Это как классический цикл
int counter = 0;
while (counter < 5) {
    System.out.println("Счётчик: " + counter);
    counter++;
}

// Но в LangGraph:
// - Явное состояние (state)
// - Узлы вместо операций
// - Условные переходы вместо if
```

## Пример 2: Агент с LLM и инструментами

Создадим настоящего агента с LangGraph:

```
START
  ↓
[Agent] ← LLM думает и решает
  ↓
Есть tool_calls?
  ├─ Да → [Tools] → возврат к [Agent]
  └─ Нет → END
```

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from typing import Annotated
import operator

# Инструмент: Калькулятор
@tool
def calculator(expression: str) -> str:
    """Полезно для математических вычислений."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Ошибка: {e}"

# Инструмент: Поиск
@tool
def search(query: str) -> str:
    """Ищет информацию (имитация)."""
    # Имитация базы знаний
    knowledge = {
        "python": "Python — высокоуровневый язык программирования",
        "java": "Java — объектно-ориентированный язык программирования",
        "ai": "AI (Artificial Intelligence) — искусственный интеллект"
    }
    
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    
    return f"Информация по запросу '{query}' не найдена"

tools = [calculator, search]

print("Инструменты созданы!")

C:\Users\Ruslan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Инструменты созданы!


In [6]:
# Состояние агента
class AgentState(TypedDict):
    # Annotated[list, operator.add] = при слиянии состояний - добавлять в список
    messages: Annotated[list, operator.add]

# Создаём LLM с инструментами
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0
)
llm_with_tools = llm.bind_tools(tools)

print("LLM готов!")

LLM готов!


In [7]:
# Узел 1: Агент (LLM думает)
def agent_node(state: AgentState) -> AgentState:
    print("\n🤔 Агент думает...")
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Узел 2: Инструменты (выполняют tool_calls)
def tools_node(state: AgentState) -> AgentState:
    print("\n🔧 Выполняем инструменты...")
    messages = state["messages"]
    last_message = messages[-1]
    
    # Словарь инструментов
    tools_dict = {tool.name: tool for tool in tools}
    
    tool_messages = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        print(f"   → {tool_name}({tool_args})")
        
        # Выполняем инструмент
        if tool_name in tools_dict:
            result = tools_dict[tool_name].invoke(tool_args)
        else:
            result = f"Неизвестный инструмент: {tool_name}"
        
        print(f"   ← {result}")
        
        # Создаём ToolMessage
        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )
    
    return {"messages": tool_messages}

# Функция решения: Продолжить или завершить?
def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    last_message = messages[-1]
    
    # Если есть tool_calls - выполняем инструменты
    if last_message.tool_calls:
        return "tools"
    # Иначе - завершаем
    else:
        return "end"

print("Узлы созданы!")

Узлы созданы!


In [8]:
# Создаём граф
workflow = StateGraph(AgentState)

# Добавляем узлы
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tools_node)

# Точка входа
workflow.set_entry_point("agent")

# Условные рёбра от агента
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",  # Если есть tool_calls → к узлу tools
        "end": END  # Иначе → завершаем
    }
)

# От tools всегда возвращаемся к agent
workflow.add_edge("tools", "agent")

# Компилируем
app = workflow.compile()

print("Граф агента создан!")

Граф агента создан!


In [9]:
# Тест 1: Простая математика
initial_state = {
    "messages": [HumanMessage(content="Сколько будет 25 * 8 + 12?")]
}

result = app.invoke(initial_state)

print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ОТВЕТ:")
print(result["messages"][-1].content)


🤔 Агент думает...



🔧 Выполняем инструменты...
   → calculator({'expression': '25 * 8 + 12'})
   ← 212

🤔 Агент думает...



ФИНАЛЬНЫЙ ОТВЕТ:
25 × 8 + 12 = 212

Давайте проверим по шагам:
1. 25 × 8 = 200
2. 200 + 12 = 212

Ответ: **212**


In [10]:
# Тест 2: Поиск информации
initial_state = {
    "messages": [HumanMessage(content="Что такое Python?")]
}

result = app.invoke(initial_state)

print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ОТВЕТ:")
print(result["messages"][-1].content)


🤔 Агент думает...



🔧 Выполняем инструменты...
   → search({'query': 'Что такое Python язык программирования'})
   ← Python — высокоуровневый язык программирования

🤔 Агент думает...



ФИНАЛЬНЫЙ ОТВЕТ:
Python — это высокоуровневый язык программирования общего назначения, который был создан Гвидо ван Россумом и впервые выпущен в 1991 году. Вот основные характеристики Python:

**Ключевые особенности:**
- **Интерпретируемый язык** — код выполняется построчно интерпретатором
- **Динамическая типизация** — типы переменных определяются во время выполнения
- **Простой и читаемый синтаксис** — использует отступы для обозначения блоков кода
- **Мультипарадигменный** — поддерживает объектно-ориентированное, функциональное и процедурное программирование
- **Кроссплатформенный** — работает на Windows, macOS, Linux и других системах

**Преимущества Python:**
1. **Простота изучения** — один из лучших языков для начинающих
2. **Большое сообщество** — множество библиотек и фреймворков
3. **Универсальность** — используется в веб-разработке, data science, машинном обучении, автоматизации и т.д.
4. **Богатая стандартная библиотека** — множество встроенных модулей

**Области применения

In [11]:
# Тест 3: Комбинация инструментов
initial_state = {
    "messages": [HumanMessage(content="""Посчитай 15 * 3, 
    а потом найди информацию про Java""")]
}

result = app.invoke(initial_state)

print("\n" + "="*50)
print("ФИНАЛЬНЫЙ ОТВЕТ:")
print(result["messages"][-1].content)


🤔 Агент думает...



🔧 Выполняем инструменты...
   → calculator({'expression': '15 * 3'})
   ← 45

🤔 Агент думает...



🔧 Выполняем инструменты...
   → search({'query': 'Java programming language информация'})
   ← Java — объектно-ориентированный язык программирования

🤔 Агент думает...



ФИНАЛЬНЫЙ ОТВЕТ:
Результаты:
1. **15 * 3 = 45**
2. **Java** — это объектно-ориентированный язык программирования, разработанный компанией Sun Microsystems (ныне принадлежит Oracle). Java известна своей кроссплатформенностью благодаря виртуальной машине Java (JVM), которая позволяет запускать Java-приложения на различных операционных системах без перекомпиляции.


### Что произошло?

**Цикл работы:**

1. **START** → Агент получает вопрос
2. **Agent Node** → LLM думает, решает использовать calculator
3. **should_continue** → Проверяет tool_calls → "tools"
4. **Tools Node** → Выполняет calculator
5. **Edge** → Возвращается к Agent Node
6. **Agent Node** → LLM видит результат, формулирует ответ
7. **should_continue** → Нет tool_calls → "end"
8. **END** → Завершение

### Преимущества LangGraph:

- ✅ Явная структура (граф переходов)
- ✅ Легко добавить новые узлы
- ✅ Можно визуализировать
- ✅ Состояние передаётся автоматически

### Java аналогия:

```java
// LangGraph — это как Workflow Engine

// Camunda BPMN:
<workflow>
  <startEvent id="start"/>
  <serviceTask id="agent" name="Think"/>
  <exclusiveGateway id="decision">
    <outgoing condition="hasToolCalls">tools</outgoing>
    <outgoing condition="else">end</outgoing>
  </exclusiveGateway>
  <serviceTask id="tools" name="Execute"/>
  <endEvent id="end"/>
</workflow>

// Или Spring State Machine:
@Configuration
class StateMachineConfig {
    @Bean
    StateMachine<States, Events> stateMachine() {
        return StateMachineBuilder.builder()
            .state("agent")
            .state("tools")
            .transition()
                .source("agent")
                .target("tools")
                .guard(hasToolCalls())
            .build();
    }
}
```

## Пример 3: Граф с ветвлениями

Создадим агента с несколькими путями:

```
START
  ↓
[Classify] ← Классифицирует запрос
  ↓
Тип запроса?
  ├─ math → [Calculator] → END
  ├─ search → [Search] → END
  └─ chat → [Chat] → END
```

In [12]:
# Состояние
class RouterState(TypedDict):
    question: str
    category: str
    answer: str

# Узел 1: Классификатор
def classify_node(state: RouterState) -> RouterState:
    question = state["question"].lower()
    
    # Простая классификация
    if any(word in question for word in ["сколько", "*", "+", "-", "/", "посчита"]):
        category = "math"
    elif any(word in question for word in ["что такое", "найди", "поиск"]):
        category = "search"
    else:
        category = "chat"
    
    print(f"📋 Категория: {category}")
    state["category"] = category
    return state

# Узел 2: Математика
def math_node(state: RouterState) -> RouterState:
    print("🔢 Обрабатываю математический запрос...")
    # Упрощённо - просто вызываем LLM с калькулятором
    state["answer"] = "Математический ответ (демо)"
    return state

# Узел 3: Поиск
def search_node(state: RouterState) -> RouterState:
    print("🔍 Ищу информацию...")
    state["answer"] = "Результат поиска (демо)"
    return state

# Узел 4: Обычный чат
def chat_node(state: RouterState) -> RouterState:
    print("💬 Обычный диалог...")
    state["answer"] = "Ответ в стиле чата (демо)"
    return state

# Функция маршрутизации
def route_question(state: RouterState) -> str:
    return state["category"]  # Возвращаем категорию

print("Узлы роутера созданы!")

Узлы роутера созданы!


In [13]:
# Создаём граф с ветвлениями
workflow = StateGraph(RouterState)

# Добавляем все узлы
workflow.add_node("classify", classify_node)
workflow.add_node("math", math_node)
workflow.add_node("search", search_node)
workflow.add_node("chat", chat_node)

# Точка входа
workflow.set_entry_point("classify")

# Условная маршрутизация
workflow.add_conditional_edges(
    "classify",
    route_question,
    {
        "math": "math",
        "search": "search",
        "chat": "chat"
    }
)

# Все пути ведут к END
workflow.add_edge("math", END)
workflow.add_edge("search", END)
workflow.add_edge("chat", END)

# Компилируем
router_app = workflow.compile()

print("Граф с роутингом создан!")

Граф с роутингом создан!


In [14]:
# Тест 1: Математика
result = router_app.invoke({"question": "Сколько будет 5 + 5?"})
print(f"\nРезультат: {result}")

📋 Категория: math
🔢 Обрабатываю математический запрос...

Результат: {'question': 'Сколько будет 5 + 5?', 'category': 'math', 'answer': 'Математический ответ (демо)'}


In [15]:
# Тест 2: Поиск
result = router_app.invoke({"question": "Что такое Python?"})
print(f"\nРезультат: {result}")

📋 Категория: search
🔍 Ищу информацию...

Результат: {'question': 'Что такое Python?', 'category': 'search', 'answer': 'Результат поиска (демо)'}


In [16]:
# Тест 3: Чат
result = router_app.invoke({"question": "Привет, как дела?"})
print(f"\nРезультат: {result}")

📋 Категория: chat
💬 Обычный диалог...

Результат: {'question': 'Привет, как дела?', 'category': 'chat', 'answer': 'Ответ в стиле чата (демо)'}


### Что произошло?

**Разные пути для разных запросов:**

1. "Сколько будет 5 + 5?" → classify → "math" → math_node → END
2. "Что такое Python?" → classify → "search" → search_node → END
3. "Привет" → classify → "chat" → chat_node → END

### Java аналогия:

```java
// LangGraph с ветвлениями — это как Strategy + Router

interface Handler {
    String handle(String question);
}

class MathHandler implements Handler { ... }
class SearchHandler implements Handler { ... }
class ChatHandler implements Handler { ... }

class Router {
    Map<String, Handler> handlers;
    
    String route(String question) {
        String category = classify(question);
        Handler handler = handlers.get(category);
        return handler.handle(question);
    }
}

// Похоже на:
// - Spring @RequestMapping с conditions
// - Apache Camel routing
// - AWS Step Functions with Choice state
```

## Сравнение подходов

| Подход | Простой Agent | LangGraph |
|--------|---------------|----------|
| Структура | Implicit (while loop) | Explicit (граф) |
| Сложность | Простые циклы | Любые графы |
| Читаемость | Средняя | Высокая |
| Отладка | Сложно | Легко |
| Визуализация | Нет | Да |
| Когда использовать | Простые задачи | Сложные workflow |

### Примеры применения:

**Простой Agent:**
- Вопрос-ответ с 1-2 инструментами
- Линейный workflow
- Быстрые прототипы

**LangGraph:**
- Многошаговые процессы
- Ветвления и условия
- Human-in-the-loop
- Workflow с проверками
- Production системы

## Итоги

### Что такое LangGraph:
- Библиотека для создания **сложных агентов**
- Явная структура через **State Graph**
- Поддержка **циклов** и **ветвлений**

### Основные концепции:
1. **State** — данные передаваемые между узлами
2. **Node** — функция обрабатывающая состояние
3. **Edge** — переход между узлами
4. **Conditional Edges** — переходы с условиями

### Java аналогии:
- State ≈ Context в паттерне State
- Node ≈ StateHandler / Service
- Graph ≈ State Machine / Workflow Engine
- LangGraph ≈ Spring State Machine / Camunda / AWS Step Functions

### Когда использовать:
- ✅ Многошаговые процессы
- ✅ Условные ветвления
- ✅ Циклы с проверками
- ✅ Сложная бизнес-логика
- ❌ Простые задачи → используй обычный Agent

### Следующие шаги:
1. **Human-in-the-loop** — добавление проверок человеком
2. **Persistence** — сохранение состояния
3. **Streaming** — потоковая обработка
4. **Production** — деплой LangGraph приложений